# 06 — Tabla comparativa final
Junta las metricas de todas las ejecuciones (03, 04, 05) en una sola tabla lista para el paper.
Correlo al final, cuando ya ejecutaste los demas notebooks.

In [ ]:
import os, datetime as _dt

def iniciar_ejecucion(nombre, config=None, base="resultados"):
    os.makedirs(base, exist_ok=True)
    nums = []
    for d in os.listdir(base):
        if d.startswith("ejecucion_") and os.path.isdir(os.path.join(base, d)):
            try: nums.append(int(d.split("_")[1]))
            except: pass
    n = max(nums)+1 if nums else 1
    carpeta = os.path.join(base, f"ejecucion_{n}")
    os.makedirs(carpeta)
    with open(os.path.join(carpeta, "config.txt"), "w", encoding="utf-8") as f:
        f.write("notebook : " + str(nombre) + "\n")
        f.write("fecha    : " + _dt.datetime.now().strftime("%Y-%m-%d %H:%M:%S") + "\n")
        if config:
            for k, v in config.items():
                f.write(str(k) + " : " + str(v) + "\n")
    print("[Ejecucion " + str(n) + "] Resultados en: " + carpeta)
    return carpeta

def log(texto, carpeta):
    print(texto)
    with open(os.path.join(carpeta, "resultados.txt"), "a", encoding="utf-8") as f:
        f.write(str(texto) + "\n")

def guardar_csv(resultados, ruta):
    if not resultados: return
    cols = list(resultados[0].keys())
    with open(ruta, "w", encoding="utf-8") as f:
        f.write(",".join(cols) + "\n")
        for r in resultados:
            f.write(",".join(str(r[c]) for c in cols) + "\n")


In [ ]:
import os, glob, csv, statistics as st
carpeta=iniciar_ejecucion("06_tabla_final")

COLS=["recall","f1","auc_roc","auc_pr","accuracy","tiempo_s"]
filas=[]
for ruta in sorted(glob.glob("resultados/ejecucion_*/metricas_por_corrida.csv")):
    cfg=os.path.join(os.path.dirname(ruta),"config.txt"); nombre=os.path.dirname(ruta)
    if os.path.exists(cfg):
        for l in open(cfg,encoding="utf-8"):
            if l.startswith("notebook"): nombre=l.split(":",1)[1].strip()
    rows=list(csv.DictReader(open(ruta,encoding="utf-8")))
    if not rows: continue
    fila={"modelo":nombre}
    for c in COLS:
        if c in rows[0]:
            fila[c]=round(st.mean(float(r[c]) for r in rows),4)
        else:
            fila[c]=""
    filas.append(fila)

if filas:
    cols=["modelo"]+COLS
    w={c:max(len(c),max(len(str(f.get(c,""))) for f in filas)) for c in cols}
    log(" | ".join(c.ljust(w[c]) for c in cols),carpeta)
    log("-+-".join("-"*w[c] for c in cols),carpeta)
    for f in filas: log(" | ".join(str(f.get(c,"")).ljust(w[c]) for c in cols),carpeta)
    guardar_csv(filas,os.path.join(carpeta,"tabla_comparativa.csv"))
    log(f"\nTabla guardada en: {carpeta}",carpeta)
else:
    log("No se encontraron metricas. Corre primero 03, 04 y 05.",carpeta)